# Advanced Resources

## Limitations of `Resource`

Up to this point the models we have used a basic `simpy.Resource`.  This has been very useful to model queues, but one potential downside to `Resource` is that it *does not allow you to model individual resource attributes or any type of complex behaviour*. For many models in healthcare, this is sufficient, but there may be instances where you need to track and control individual resources. For example, ambulances, or different types of staff. In this notebook we will explore how to add more complex behaviour using the `Store` and `FilterStore` objects provided by `simpy`.

🎓 The good news is that both `Store` and `FilterStore` are easy to use.  Usually this is within the context of a complex simulation, but we will keep our models simple here and focus on how to use them.

## When to use each SimPy resource type

| Type | Best used for | Key idea |
|---|---|---|
| `Resource` | Identical servers | Only capacity matters. |
| `Store` | Individual objects | Retrieve the next available object. |
| `FilterStore` | Individual objects with selection rules | Retrieve an object matching a condition. |


## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import itertools
import simpy
import math

In [2]:
# to reduce code these classes can be found in distribution.py
# you can also pip `install sim-tools`
from distributions import (
    Exponential, 
    DiscreteEmpirical
)

from sim_utility import set_trace, trace, spawn_seeds

## 2. PART 1: Using a `simpy.Store`

A `Store` can be thought of as a container that holds Python objects. We can `put()` objects into the store and later `get()` them back out again, usually on a First In First Out (FIFO) basis.

This is useful when resources are not all identical. With a normal `simpy.Resource`, we know how many units are available, but we do not know which specific unit is being used. With a `Store`, we can model individual resources with their own attributes.

We create a `Store` as follows (`env` is a `simpy.Environment`):

```python
store = simpy.Store(env, capacity=2)
```
We have to `put` something in the store. Let's take a very simple example where people experiencing a medical emergency are assigned either a Rapid Response Vehicle or a "normal" Type 1 Ambulance.  A process flow describing use of the `Store` is below.

<img src="img/two_resource_types.png" alt="model image" width="800">

This is a two part process. We first create the instances of ambulances and then we call `Store.put()`


```python
ambulances = [
    Ambulance(ambulance_id=1, vehicle_type="rrv"), 
    Ambulance(ambulance_id=2, vehicle_type="type_1")
]

# loop through list of ambulances and put them in the store object
for amb in ambulances:
    store.put(amb)

```

To get an an ambulance we used the `get()` method along with the `yield` keyword.

```python
ambulance = yield store.get()
```

Like normal `Resource` objects the use of `yield` means that we can simulate queues when no `Ambulance` objects are left in the store.

The code below implements the full model. In the model we will track

* The utilisation and number of emergencies handled by each individual vehicle.
* Overall patient waiting time and associated statistics.
 
### 2.1 Parameters

In [3]:
NUM_AMBULANCES = 10
FLEET = {
    "rrv": 4,
    "type_1": 6,
}

RRV_SERVICE_TIME   = 50.0   # minutes
TYPE1_SERVICE_TIME = 65.0   
RUN_LENGTH         = 1_000  # minutes
RANDOM_SEED        = 42

# exponential IAT 
MEAN_INTERARRIVAL = 6

### 2.2 Entity classes

We will model Ambulances using a simple Python class called `Ambulance`. For simplicity we will give the ambulance an attribute called `vehicle_type`. We will use that to look up the service time distribution from a Python dictionary parameter.

In [4]:
class Ambulance:
    """
    An ambulance resource

    Parameters:
    ----------
    ambulance_id: int
        Unique id of the ambulance
    vehicle_type: str
        Set to either "rrv" or "type_1"
    """
    def __init__(self, ambulance_id: int, vehicle_type: str):
        self.ambulance_id = ambulance_id

        # vehicle type can be "rrv" or "type_1"
        self.vehicle_type = vehicle_type
        
        self.total_jobs   = 0
        # cumulative busy time
        self.total_busy   = 0.0   

        self.emojis = {
            "rrv": "🏎️",
            "type_1": "🚑"
        }
        
        self.emoji = self.emojis[self.vehicle_type]
        
    def __repr__(self):
        """
        A text representation of the ambulance to help debugging
        """
        return f"Ambulance({self.ambulance_id}, {self.emoji})"

In [5]:
class Patient:
    """
    A class to hold patient attributes

    Parameters
    ----------
    patient_id: int
        Unique patient id

    arrival_time: float
        Time of arrival to the simulation    
    """
    def __init__(self, patient_id: int, arrival_time: float):
        self.patient_id   = patient_id
        self.arrival_time = arrival_time

    def __repr__(self):
        """
        A text representation of the Patient for debug
        """
        return f"Patient({self.patient_id})"

## 2.3 Ambulance dispatch and service process

The code looks a little different from when using a standard `Resource` because we do not use a `with` context manager when using a `Store`. This means we need to explicitly "release" an `Ambulance` by putting it back into the `Store`,

The function `dispatch_ambulance` is a `simpy` process that accepts a `dict` called `dists`. It contains a service time distribution for each type of ambulance i.e. "rrv" and "type_1".  For example if we had a rapid response vehicle we could sample a service time as follows:

For example:

```python
ambulance = Ambulance(1, "rrv")

# vehicle type is a str with value "rrv" It returns a Exponential(50)
service_dist = dists[ambulance.vehicle_type]
service_time = service_dist.sample()

```

In [6]:
def dispatch_ambulance(
    env: simpy.Environment,
    store: simpy.Store,
    patient: Patient,
    dists: dict,
    log: dict
) -> None:
    """Simulate ambulance dispatch and service: 
    
    1. queues a patient up for the next free Ambulance (FIFO), 
    2. simulates service (as a single distribution), 
    3. returns the Ambulance to the store.

    Parameters:
    ----------
    env: simpy.Environment
        The simpy environment for the simulation
    store: simpy.Store
        A store of Ambulance objects
    dists: dict
        Contains the "service" distribution
    log: dict
        Audit dictionary
    """

    # Wait for an available Ambulance
    # note we `get()` an Ambulance from the store
    ambulance: Ambulance = yield store.get()

    wait_time = env.now - patient.arrival_time
    log["wait_times"].append(wait_time)

    # Service varies by vehicle type (travel + on-scene + return)
    service_dist = dists[ambulance.vehicle_type]
    service_time = service_dist.sample()

    # debug
    trace(
        f"{env.now:.1f}: {ambulance.emoji} {ambulance.ambulance_id} → {patient} "
        f"(waited {wait_time:.1f} min, service {service_time:.1f} min)"
    )

    yield env.timeout(service_time)

    # Update ambulance stats and return (put) to store
    ambulance.total_jobs += 1
    ambulance.total_busy += service_time
    store.put(ambulance)

    log["service_times"].append(service_time)
    log["assignments"].append(
        (patient.patient_id, ambulance.ambulance_id, ambulance.vehicle_type)
    )

### 2.4 Patient arrival generator

Our patient generator process follows the simple template approach we have used before: an infinite loop where we sample the time until the next arrival and then create and schedule new `dispatch_ambulance` process.

In [7]:
def patient_arrivals_generator(
    env: simpy.Environment,
    store: simpy.Store,
    dists: dict,
    log: dict
) -> None:
    """
    Arrival process for patients to the ambulance sim.

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    store: simpy.Store
        A store of Ambulance objects

    dists: dict
        Contains "arrival" and "service" distributions

    log: dict
        Results dictionary
    """
    for patient_id in itertools.count(start=1):

        # time until next patient arrival
        inter_arrival_time = dists["arrival"].sample()
        yield env.timeout(inter_arrival_time)

        log["n_arrivals"] += 1
        patient = Patient(patient_id, env.now)

        # debug info
        trace(f"{env.now:.1f}: 📞 {patient}")

        # create ambulance dispatch + service process
        env.process(dispatch_ambulance(env, store, patient, dists, log))

### 2.5 Single run function

The `single_run` function creates our dictionary of distributions, ambulances, `Store` resource and runs the simulation returning a log of results and the ambulances that we will use the pass to a simple function called `results_summary` that will print out some formatted statistics.

In [8]:
def single_run(
    mean_iat: float = MEAN_INTERARRIVAL,
    mean_rrv_service: float = RRV_SERVICE_TIME,
    mean_type_1_service: float = TYPE1_SERVICE_TIME,
    n_ambulances: int = NUM_AMBULANCES,
    fleet: dict = FLEET,
    run_length: float = RUN_LENGTH, 
    random_seed: int = 1
):
    """
    Set up and perform a single replication of the MMS model
    """

    # generate 3 rng seeds
    seeds = spawn_seeds(n_streams=3, main_seed=random_seed)
    
    # 1. distribution objects
    dists = {
        "arrival": Exponential(mean_iat, random_seed=seeds[0]),
        "rrv": Exponential(mean_rrv_service, random_seed=seeds[1]),
        "type_1": Exponential(mean_type_1_service, random_seed=seeds[2]),
    }

    # 2. simpy environment 
    env = simpy.Environment()

    # 3. Initialise Store
    # 3.1 Create empty Store with sufficient slots
    store = simpy.Store(env, capacity=n_ambulances)

    # 3.2 Create Ambulance objects
    ambulances = []
    ambulance_id = 0
    
    for vehicle_type, count in fleet.items():
        for _ in range(count):
            ambulance_id += 1
            ambulances.append(Ambulance(ambulance_id, vehicle_type))
    
    # 3.3 `put` Ambulance objects into the store
    for amb in ambulances:
        store.put(amb)

    # 4. results dictionary
    log = {"n_arrivals": 0, "wait_times": [], "service_times": [], "assignments": []}

    env.process(patient_arrivals_generator(env, store, dists, log))
    env.run(until=run_length)

    return ambulances, log

In [9]:
def results_summary(
    log: dict,
    ambulances: list[Ambulance],
    run_length: float,
    mean_iat: float
):
    waits = np.array(log["wait_times"])
    n_served = len(waits)

    print("\n" + "═" * 55)
    print(f" Mean inter-arrival : {mean_iat:.2f} min")
    print("─" * 55)
    print(f" Patients arrived   : {log['n_arrivals']}")
    print(f" Patients served    : {n_served}")

    if n_served > 0:
        print(f" Mean wait time     : {waits.mean():.2f} min")
        print(f" P(wait > 0)        : {(waits > 0).mean():.2%}")
        print(f" 95th pct wait      : {np.percentile(waits, 95):.2f} min")
    else:
        print(" No patients were served.")

    print("─" * 55)
    print(f" {'Ambulance':<15} {'Jobs':>6} {'Utilisation':>12}")
    print("─" * 55)

    for amb in ambulances:
        util = amb.total_busy / run_length
        print(f" {amb.emoji} {amb.ambulance_id:<5} {amb.total_jobs:>6} {util:>11.2%}")

    print("═" * 55)

### 2.6 Run the model and view results 

Use `set_trace` to toggle the printing of the model event debug on and off. 

In [10]:
set_trace(False)

ambulances, log = single_run(run_length=RUN_LENGTH, random_seed=42)
results_summary(log, ambulances, run_length=RUN_LENGTH, mean_iat=MEAN_INTERARRIVAL)

Simulation tracing set to: False

═══════════════════════════════════════════════════════
 Mean inter-arrival : 6.00 min
───────────────────────────────────────────────────────
 Patients arrived   : 168
 Patients served    : 157
 Mean wait time     : 38.29 min
 P(wait > 0)        : 78.98%
 95th pct wait      : 90.39 min
───────────────────────────────────────────────────────
 Ambulance         Jobs  Utilisation
───────────────────────────────────────────────────────
 🏎️ 1         19      79.09%
 🏎️ 2         18      86.62%
 🏎️ 3         13      85.86%
 🏎️ 4          9      79.45%
 🚑 5         10      82.41%
 🚑 6         18      81.24%
 🚑 7         15      87.30%
 🚑 8         17      90.95%
 🚑 9         15      89.03%
 🚑 10        13      86.19%
═══════════════════════════════════════════════════════


## 3. PART 2: Using a `FilterStore`

In this example we will modify the ambulance dispatch simulation so that patients and ambulances are located within one of five **nodes** that are specified by coordinates.

| ID | Label | Coordinates |
|---|---|---|
| 0 | South-West | `(0, 0)` |
| 1 | South-East | `(4, 0)` |
| 2 | North-West | `(0, 4)` |
| 3 | North-East | `(4, 4)` |
| 4 | Centre | `(2, 2)` |
| 5 | Hospital | `(2, 0)` |


Patients in need will be assigned the closest available ambulance. This is why we use a `FilterStore`.  We iterate through items in the store to locate the closest ambulance.

An assigned ambulance has to travel to the patient (constant speed), pick them up (Exponential), travel to the hospital, and then travel back to their home node.

If no ambulances are available they are assigned the next available ambulance when it has returned to its home node.

### 3.1 Geographic information

In [11]:
# All locations share a single distance matrix: nodes 0-4 plus hospital (5)
HOSPITAL_ID = 5

LOCATIONS = {
    0: (0.0, 0.0),   # South-West
    1: (4.0, 0.0),   # South-East
    2: (0.0, 4.0),   # North-West
    3: (4.0, 4.0),   # North-East
    4: (2.0, 2.0),   # Centre
    HOSPITAL_ID: (2.0, 0.0),  # Hospital (south-centre)
}

# nodes patients can appear in
PATIENT_NODES = [0, 1, 2, 3, 4]   
NODE_PROBS = [0.15, 0.25, 0.20, 0.20, 0.20]

# Pre-compute ALL pairwise distances before simulation
DISTANCES = {
    (i, j): math.dist(LOCATIONS[i], LOCATIONS[j])
    for i in LOCATIONS
    for j in LOCATIONS
}

### 3.2 Parameters

In [12]:
NUM_AMBULANCES    = 10
TRAVEL_SPEED      = 0.25  # distance units per minute
MEAN_SCENE_TIME   = 20.0  
MEAN_INTERARRIVAL = 6    
RUN_LENGTH        = 1_000
RANDOM_SEED       = 42

# Ambulance positioning parameter
# 2 ambulances stationed at each of the 5 nodes
AMBULANCE_HOME_NODES = [
    node
    for node in PATIENT_NODES
    for _ in range(NUM_AMBULANCES // len(PATIENT_NODES))
]

### 3.3 Entity Classes

The `Ambulance` and `Patient` classes have been slightly modified to include a `node` attribute. For simplicity all ambulances have the same type

In [13]:
class Ambulance:
    def __init__(self, ambulance_id: int, home_node: int):
        self.ambulance_id = ambulance_id
        # modification = a home node or 'base' for the ambulance
        self.home_node    = home_node
        self.total_jobs   = 0
        self.total_busy   = 0.0

    def __repr__(self):
        return f"Ambulance(id={self.ambulance_id}, node={self.home_node})"

In [14]:
class Patient:
    def __init__(self, patient_id: int, arrival_time: float, node: int):
        self.patient_id  = patient_id
        self.arrival_time = arrival_time
        self.node = node

    def __repr__(self):
        return f"Patient(id={self.patient_id}, node={self.node})"

### 3.4 Travel functions

In [15]:
def travel_time(loc_a: int, loc_b: int) -> float:
    """Minutes to travel between any two location IDs."""
    return DISTANCES[loc_a, loc_b] / TRAVEL_SPEED

In [16]:
def closest_ambulance_id(
    available: list[Ambulance],
    patient_node: int
) -> int | None:
    """
    Finds the closest ambulance and return the ID.
    Note this code is not designed for efficiency. 
    """

    # if list of ambulance is empty
    if not available:
        return None

    # return the closest ambulance id
    return min(available, key=lambda a: DISTANCES[a.home_node, patient_node]).ambulance_id

### 3.5 Modified simpy processes

The biggest update to our ambulance dispatch function is that it will now use a `FilterStore` to take account of `Patient` and `Ambulance` node location.

Our code also needs to be able to handle a situation where all ambulances are in service.  In these instances a `Patient` would just wait for the next available ambulance on a FIFO basis.  

Examples. Let's assume ambulance 2 is the closest to the patient. We `get` that ambulance from the filter store like so

```python
best_id = 2
ambulance = yield store.get(lambda a: a.ambulance_id == best_id)
```

In an instance where there are no ambulances available we can force the `FilterStore` to work on a FIFO basis like so

```python
ambulance = yield store.get(lambda a: True)
```

In [17]:
def dispatch_ambulance(env, store, patient, dists, log):
    """Modified ambulance dispatch process
    """
   
    # find the closest ambulance in the store
    # note we pass store.items which is a list of Ambulance objects
    # it may be empty! This means all ambulances are in use.
    best_id = closest_ambulance_id(store.items, patient.node)

    # filter store code
    if best_id is not None:
        # if an ambulance is available get the closest ambulance from the store
        ambulance = yield store.get(lambda a: a.ambulance_id == best_id)
    else:
        # otherwise just wait for the next available ambulance. FIFO
        # we are forcing a FilterStore to behave like a standard Store
        ambulance = yield store.get(lambda a: True)

    wait_time = env.now - patient.arrival_time

    # Leg 1: travel from home node to patient 
    t_to_patient = travel_time(ambulance.home_node, patient.node)

    # Leg 2: on scene
    scene_time = dists["on_scene"].sample()

    # Leg 3: transport patient to hospital
    t_to_hospital = travel_time(patient.node, HOSPITAL_ID)

    # Leg 4: return to home base 
    t_to_base = travel_time(HOSPITAL_ID, ambulance.home_node)

    # total turnaround time
    total_service = t_to_patient + scene_time + t_to_hospital + t_to_base

    # debug
    trace(
        f"{env.now:.1f}: 🚑 {ambulance.ambulance_id} → {patient} "
        f"(waited {wait_time:.1f} min, service {total_service:.1f} min)"
    )

    yield env.timeout(total_service)
    
    ambulance.total_jobs += 1
    ambulance.total_busy += total_service
    store.put(ambulance)

    log["wait_times"].append(wait_time)

    # create row in the assignments table
    log["assignments"].append(
        dict(patient_id=patient.patient_id,
             patient_node=patient.node,
             ambulance_id=ambulance.ambulance_id,
             ambulance_node=ambulance.home_node,
             dispatch_dist=DISTANCES[ambulance.home_node, patient.node],
             t_to_patient=t_to_patient,
             scene_time=scene_time,
             t_to_hospital=t_to_hospital,
             t_to_base=t_to_base,
             total_service=total_service,
             immediate_dispatch=best_id is not None,
             wait=wait_time)
    )

In [18]:
def patient_arrivals_generator(
    env: simpy.Environment,
    store: simpy.Store,
    dists: dict,
    log: dict
) -> None:       
    """Modified Arrival process for patients to the ambulance sim
    Patient now includes arrival node"""
    for patient_id in itertools.count(start=1):

        # time until next patient arrival
        inter_arrival_time = dists["arrival"].sample()
        yield env.timeout(inter_arrival_time)

        log["n_arrivals"] += 1
        node  = dists["arrival_node"].sample()
        patient = Patient(patient_id, env.now, node)

        # debug info
        trace(f"{env.now:.1f}: 📞 {patient}")

        # create ambulance dispatch + service process
        env.process(dispatch_ambulance(env, store, patient, dists, log))

In [23]:
def single_run(
    mean_iat: float = MEAN_INTERARRIVAL,
    mean_on_scene: float = MEAN_SCENE_TIME,
    n_ambulances: int = NUM_AMBULANCES,
    run_length: float = RUN_LENGTH, 
    random_seed: int = 42
):
    """
    Set up and perform a single replication of the MMS model
    """

    # generate 3 rng seeds
    seeds = spawn_seeds(n_streams=3, main_seed=random_seed)
    
    # 1. distribution objects
    # We have included a new arrival_node an on_scene dists
    dists = {
        "arrival": Exponential(mean_iat, random_seed=seeds[0]),
        "arrival_node": DiscreteEmpirical(
            values=[0, 1, 2, 3, 4], 
            freq=[p * 100 for p in NODE_PROBS], # need to freqs not probs
            random_seed=seeds[1]),
        "on_scene": Exponential(mean_on_scene, random_seed=seeds[2]),
    }

    # 2. simpy environment 
    env = simpy.Environment()

    # 3. Initialise Store
    # 3.1 Create empty Store with sufficient slots
    store = simpy.FilterStore(env, capacity=n_ambulances)

    # 3.2 Create Ambulance objects now with home nodes
    ambulances = [Ambulance(i + 1, home) for i, home in enumerate(AMBULANCE_HOME_NODES)]

    # 3.3 `put` Ambulance objects into the filter store
    for amb in ambulances:
        store.put(amb)

    # 4. results dictionary
    log = {"n_arrivals": 0, "wait_times": [], "service_times": [], "assignments": []}

    env.process(patient_arrivals_generator(env, store, dists, log))
    env.run(until=run_length)

    return ambulances, log

In [20]:
def single_run_summary(assignments, ambulances, run_length):
    waits = assignments["wait"].to_numpy()
    
    # immediate versus delayed dispatches of Ambulance
    immediate  = assignments[assignments["immediate_dispatch"]]
    delayed = assignments[~assignments["immediate_dispatch"]]
    
    print("\n" + "═" * 65)
    print(f"  Patients served            : {len(assignments)}")
    print(f"  Immediate Dispatch         : {len(immediate)}   ({len(immediate)/len(assignments):.1%})")
    print(f"  Delayed Dispatch           : {len(delayed)}  ({len(delayed)/len(assignments):.1%})")
    print(f"  Overall mean wait (min)    : {waits.mean():.2f}")
    print("─" * 65)
    print(f"  {'':<30} {'Ambulance Dispatch':>20}")
    print(f"  {'Metric':<30} {'Immediate':>10} {'Delayed':>10}")
    print("─" * 65)
    for metric, col in [("Mean wait (min)",         "wait"),
                        ("Mean dispatch dist",       "dispatch_dist"),
                        ("Mean travel to patient",   "t_to_patient"),
                        ("Mean scene time",          "scene_time"),
                        ("Mean travel to hospital",  "t_to_hospital"),
                        ("Mean return to base",      "t_to_base"),
                        ("Mean total service",       "total_service")]:
        c = immediate[col].mean()  if len(immediate)  else float("nan")
        f = delayed[col].mean() if len(delayed) else float("nan")
    
        print(f"  {metric:<30} {c:>10.2f} {f:>10.2f}")
    print("─" * 65)
    print(f"  {'Ambulance':<14} {'Node':>10} {'Jobs':>8} {'Util':>10}")
    print("─" * 65)
    for amb in ambulances:
        print(f"  Ambulance {amb.ambulance_id:<4}      "
              f"{amb.home_node:>5}  {amb.total_jobs:>6}    "
              f"{amb.total_busy / run_length:>8.2%}")
    print("═" * 65)

In [21]:
set_trace(False)

# run model
ambulances, log = single_run(run_length=RUN_LENGTH, random_seed=42)

# get all of the patient to ambulance assignment details
assignments = pd.DataFrame(log["assignments"])
assignments.tail()

Simulation tracing set to: False


,patient_id,patient_node,ambulance_id,ambulance_node,dispatch_dist,t_to_patient,scene_time,t_to_hospital,t_to_base,total_service,immediate_dispatch,wait
152,152,0,10,4,2.828427,11.313708,17.392233,8.000000,8.000000,44.705942,True,0.000000
153,149,3,8,3,0.000000,0.000000,36.456053,17.888544,17.888544,72.233141,True,0.000000
154,155,3,6,2,4.000000,16.000000,0.483107,17.888544,17.888544,52.260194,True,0.000000
155,160,1,2,0,4.000000,16.000000,0.067088,8.000000,8.000000,32.067088,False,13.713547
156,151,1,3,1,0.000000,0.000000,55.866957,8.000000,8.000000,71.866957,True,0.000000


In [22]:
set_trace(False)
ambulances, log = single_run(random_seed=42)
single_run_summary(assignments, ambulances, RUN_LENGTH)

Simulation tracing set to: False

═════════════════════════════════════════════════════════════════
  Patients served            : 157
  Immediate Dispatch         : 73   (46.5%)
  Delayed Dispatch           : 84  (53.5%)
  Overall mean wait (min)    : 13.34
─────────────────────────────────────────────────────────────────
                                   Ambulance Dispatch
  Metric                          Immediate    Delayed
─────────────────────────────────────────────────────────────────
  Mean wait (min)                      0.00      24.93
  Mean dispatch dist                   1.35       2.79
  Mean travel to patient               5.38      11.16
  Mean scene time                     20.32      19.71
  Mean travel to hospital             11.12      11.30
  Mean return to base                 11.25      11.41
  Mean total service                  48.07      53.58
─────────────────────────────────────────────────────────────────
  Ambulance            Node     Jobs       Util
─